In [ ]:
# ============================================================
# Cell 0 - Import Packages and Load BTMS Model
# ============================================================

import os
import sys
import io
import contextlib
import itertools
import importlib

import numpy as np
import pandas as pd
import CoolProp.CoolProp as CP

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side
from openpyxl.utils import get_column_letter

PROJECT_ROOT = os.path.abspath("../..")

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import lib.BTMS_model as BTMS_model

BTMS_model = importlib.reload(BTMS_model)

print(f'Project root: {PROJECT_ROOT}')
print(f'BTMS_model loaded from: {BTMS_model.__file__}')

Project root: d:\Users\liu\Desktop\eBATS_scanned_2
BTMS_model loaded from: d:\Users\liu\Desktop\eBATS_scanned_2\lib\BTMS_model.py


In [ ]:
# ============================================================
# Cell 1 - Result-File Naming Configuration
# ============================================================

CASE_ID = 'C0003'
REVISION = 'R02'
RESULT_ID = f'{CASE_ID}{REVISION}'

SCENARIO_CODE = 'PARK25'
BTMS_CODE = 'LC'
MODEL_CODE = '1D'
TASK_CODE = 'SCAN'

RESULT_BASENAME = f'{RESULT_ID}_{SCENARIO_CODE}_{BTMS_CODE}_{MODEL_CODE}_{TASK_CODE}'
NOTEBOOK_NAME = f'{RESULT_BASENAME}.ipynb'
RESULT_XLSX_NAME = f'{RESULT_BASENAME}.xlsx'

OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'out', CASE_ID)
os.makedirs(OUTPUT_DIR, exist_ok=True)

RESULT_XLSX_PATH = os.path.join(OUTPUT_DIR, RESULT_XLSX_NAME)

print(f'Result basename: {RESULT_BASENAME}')
print(f'Notebook file: {NOTEBOOK_NAME}')
print(f'Official Excel result: {RESULT_XLSX_PATH}')


Result basename: C0003R02_PARK25_LC_1D_SCAN
Notebook file: C0003R02_PARK25_LC_1D_SCAN.ipynb
Official Excel result: d:\Users\liu\Desktop\eBATS_scanned_2\out\C0003\C0003R02_PARK25_LC_1D_SCAN.xlsx


In [ ]:
# ============================================================
# Cell 2 - Load Battery Heat-Generation Data
# ============================================================

# Keep the physical simulation duration unchanged, while using the original time step.
SIM_DURATION_S = 1920.0   # s, physical mission duration
dt = 1.0                 # s, integration time step

NUM_STEPS = int(round(SIM_DURATION_S / dt))
if not np.isclose(NUM_STEPS * dt, SIM_DURATION_S):
    raise ValueError('SIM_DURATION_S must be an integer multiple of dt.')

# Keep the original variable name for downstream compatibility.
# Here SIM_TIME_S means the number of numerical time steps, not the physical duration.
SIM_TIME_S = NUM_STEPS

file_name = os.path.join(PROJECT_ROOT, 'data', 'MD05E070207A1  data_power gen_single cell_Liu_20260421.xlsx')

if not os.path.isfile(file_name):
    raise FileNotFoundError(
        f'Heat-generation data file not found: {file_name}\n'
        'Place the Excel file in the project data folder.'
    )
    
df = pd.read_excel(file_name, sheet_name='Sheet1')
power_generation_data_1s = df['Qbat(W)'].dropna().to_numpy(dtype=float)

# The source Qbat data are treated as 1-s samples. For dt = 1.0 s,
# each 1-s heat-generation value is used for one numerical time step.
time_s = np.arange(SIM_TIME_S + 1) * dt
power_indices = np.floor(time_s[:-1]).astype(int)
power_indices = np.clip(power_indices, 0, len(power_generation_data_1s) - 1)
power_generation_data = power_generation_data_1s[power_indices]

print(f'Simulation duration: {SIM_DURATION_S:.1f} s')
print(f'Time step dt: {dt:.3f} s')
print(f'Number of numerical steps: {SIM_TIME_S}')
print(f'Q_gen array length: {len(power_generation_data)}')

Simulation duration: 1920.0 s
Time step dt: 1.000 s
Number of numerical steps: 1920
Q_gen array length: 1920


In [ ]:
# ============================================================
# Cell 3 - Model Settings and SCAN1 Parameter Ranges
# ============================================================



BATTERY_PROPS = {
    'm_bat': 48e-3,          # kg
    'cp_bat': 830.0,         # J/(kg K)
    'D_bat': 18e-3,          # m
    'H_bat': 65e-3,          # m
}


MODULE_LAYOUT = {
    'N_r': 20,               # Control volumes along the coolant-flow direction
    'N_c': 16,               # Battery columns in the transverse direction
}



BASELINE_CASE = {
    'T_water': 25.0,         # degC
    'm_dot_total': 0.06106836,  # kg/s
    'num_channel': 20,
}


LIQUID_SETTINGS = {
    'fluid': 'Water',
    'p_water': 101325.0,

    # Fixed rectangular-channel geometry
    'W_channel': 8.0e-3,      # 8 mm
    'H_channel': 3.0e-3,      # 3 mm
    'L_channel': 0.380,       # 380 mm
    'S_channel': 17.77e-3,    # 17.77 mm clear spacing

    # Cold-plate geometry
    'delta_plate_coolant': 2.0e-3,

    # Cold-plate material properties
    'k_plate': 200.0,        # W/(m K)
    'rho_plate': 2719.0,     # kg/m3

    # Pump and additional-component settings
    'pump_efficiency': 0.35,
    'K_minor': 0.0,
    'm_pump': 0.0,           # kg
    'm_pipe': 0.0,           # kg
}



SOLVER_SETTINGS = {
    'solver_tol': 1e-6,
    'solver_maxiter': 1000,
}



mission_stages = [
    ('takeoff',       0.0,    6.0),
    ('climb',         6.0,   36.0),
    ('transition1',  36.0,  180.0),
    ('cruise',      180.0, 1560.0),
    ('transition2',1560.0, 1704.0),
    ('descent',    1704.0, 1734.0),
    ('hover',      1734.0, 1914.0),
    ('landing',    1914.0, 1920.0),
]



SCAN_VALUES = {
    'T_water_list': np.array([
        20.0,
        25.0,
        30.0,
        35.0,
    ], dtype=float),

    # Total mass flow rate of the complete cooling system
    'm_dot_total_list': np.array([
        0.045,
        0.050,
        0.055,
        0.060,
        0.065,
        0.070,
    ], dtype=float),

    # N_c / num_cell_seg gives the number of parallel channels
    'num_cell_seg_list': np.array([
        16.0 / 12.0,   # 12 parallel channels
        16.0 / 16.0,   # 16 parallel channels
        16.0 / 20.0,   # 20 parallel channels, baseline case
        16.0 / 24.0,   # 24 parallel channels
        16.0 / 28.0,   # 28 parallel channels
        16.0 / 32.0,   # 32 parallel channels
    ], dtype=float),
}

# ==========================================      ==================
# Variables used by downstream calculations
# ============================================================

m_bat = BATTERY_PROPS['m_bat']
cp_bat = BATTERY_PROPS['cp_bat']
D_bat = BATTERY_PROPS['D_bat']
H_bat = BATTERY_PROPS['H_bat']
A_battery = np.pi * D_bat * H_bat

N_r = MODULE_LAYOUT['N_r']
N_c = MODULE_LAYOUT['N_c']
num_seg = N_r

fluid = LIQUID_SETTINGS['fluid']
p_water = LIQUID_SETTINGS['p_water']

W_channel = LIQUID_SETTINGS['W_channel']
H_channel = LIQUID_SETTINGS['H_channel']
L_channel = LIQUID_SETTINGS['L_channel']
S_channel = LIQUID_SETTINGS['S_channel']
delta_plate_coolant = LIQUID_SETTINGS['delta_plate_coolant']

k_plate = LIQUID_SETTINGS['k_plate']
rho_plate = LIQUID_SETTINGS['rho_plate']

pump_efficiency = LIQUID_SETTINGS['pump_efficiency']
K_minor = LIQUID_SETTINGS['K_minor']
m_pump = LIQUID_SETTINGS['m_pump']
m_pipe = LIQUID_SETTINGS['m_pipe']

# Total cold-plate thickness
H_plate = H_channel + delta_plate_coolant

# Liquid cooling operates during the full mission
pump_operation_time = SIM_DURATION_S

solver_tol = SOLVER_SETTINGS['solver_tol']
solver_maxiter = SOLVER_SETTINGS['solver_maxiter']

T_water_list = SCAN_VALUES['T_water_list']
m_dot_total_list = SCAN_VALUES['m_dot_total_list']
num_cell_seg_list = SCAN_VALUES['num_cell_seg_list']

print('Parameter-scan settings loaded.')
print(f'Channel width = {W_channel * 1000.0:.2f} mm')
print(f'Channel height = {H_channel * 1000.0:.2f} mm')
print(f'Channel length = {L_channel * 1000.0:.2f} mm')
print(f'Channel spacing = {S_channel * 1000.0:.2f} mm')

print(f'Plate-to-coolant distance = {delta_plate_coolant * 1000.0:.2f} mm')
print(f'Cold-plate thickness = {H_plate * 1000.0:.2f} mm')
print(f'Baseline total mass flow rate = {BASELINE_CASE["m_dot_total"]:.8f} kg/s')
print(f'Baseline inlet-water temperature = {BASELINE_CASE["T_water"]:.2f} degC')
print(f'Baseline channel number = {BASELINE_CASE["num_channel"]}')



Parameter-scan settings loaded.
Channel width = 8.00 mm
Channel height = 3.00 mm
Channel length = 380.00 mm
Channel spacing = 17.77 mm
Plate-to-coolant distance = 2.00 mm
Cold-plate thickness = 5.00 mm
Baseline total mass flow rate = 0.06106836 kg/s
Baseline inlet-water temperature = 25.00 degC
Baseline channel number = 20


In [ ]:
# ============================================================
# Cell 4 - Build Parameter-Scan Cases
# ============================================================

def build_scan_cases(
    T_water_list,
    m_dot_total_list,
    num_cell_seg_list,
):
    """
    Build all combinations of coolant inlet temperature,
    total coolant mass flow rate, and equivalent cells
    represented by one parallel channel.
    """

    cases = []

    for case_count, (
        T_water,
        m_dot_total,
        num_cell_seg,
    ) in enumerate(
        itertools.product(
            T_water_list,
            m_dot_total_list,
            num_cell_seg_list,
        ),
        start=1,
    ):
        cases.append({
            'case_id': f'{CASE_ID}_S{case_count:03d}',
            'T_water': float(T_water),
            'm_dot_total': float(m_dot_total),
            'num_cell_seg': float(num_cell_seg),
        })

    return cases, pd.DataFrame(cases)


scan_cases, scan_cases_df = build_scan_cases(
    T_water_list,
    m_dot_total_list,
    num_cell_seg_list,
)

print(f'Total cases: {len(scan_cases)}')

print(
    'Inlet water temperatures: '
    f'{scan_cases_df["T_water"].drop_duplicates().to_numpy()} degC'
)

print(
    'Total coolant mass flow rates: '
    f'{scan_cases_df["m_dot_total"].drop_duplicates().to_numpy()} kg/s'
)

print(
    'Equivalent cells per channel: '
    f'{scan_cases_df["num_cell_seg"].drop_duplicates().to_numpy()}'
)

scan_cases_df.head()

Total cases: 144
Inlet water temperatures: [20. 25. 30. 35.] degC
Total coolant mass flow rates: [0.045 0.05  0.055 0.06  0.065 0.07 ] kg/s
Equivalent cells per channel: [1.33333333 1.         0.8        0.66666667 0.57142857 0.5       ]


,case_id,T_water,m_dot_total,num_cell_seg
0,C0003_S001,20.0,0.045,1.333333
1,C0003_S002,20.0,0.045,1.000000
2,C0003_S003,20.0,0.045,0.800000
3,C0003_S004,20.0,0.045,0.666667
4,C0003_S005,20.0,0.045,0.571429


In [ ]:
# ============================================================
# Cell 5 - Calculate Coolant Properties for SCAN1
# ============================================================

water_props_cache = {}

for T_water in sorted(scan_cases_df['T_water'].unique()):
    T_water = float(T_water)
    T_water_K = T_water + 273.15

    mu_water = CP.PropsSI('V', 'T', T_water_K, 'P', p_water, fluid)
    rho_water = CP.PropsSI('D', 'T', T_water_K, 'P', p_water, fluid)
    cp_water = CP.PropsSI('C', 'T', T_water_K, 'P', p_water, fluid)
    k_water = CP.PropsSI('L', 'T', T_water_K, 'P', p_water, fluid)
    Pr_water = cp_water * mu_water / k_water

    water_props_cache[T_water] = {
        'mu_water': mu_water,
        'rho_water': rho_water,
        'cp_water': cp_water,
        'k_water': k_water,
        'Pr_water': Pr_water,
    }

print(f'Water properties calculated for {len(water_props_cache)} inlet temperatures.')

Water properties calculated for 4 inlet temperatures.


In [ ]:
# ============================================================
# Cell 6 - Run One Parameter-Scan Case
# ============================================================

def run_one_case(case_id, T_water, m_dot_total, num_cell_seg):
    water_props = water_props_cache[float(T_water)]
    if m_dot_total <= 0:
        raise ValueError(
            'm_dot_total must be positive.'
            )
    if num_cell_seg <= 0:
        raise ValueError(
            'num_cell_seg must be positive.'
            )
    # ---------------------------------------------------------
    # Number of parallel channels and per-channel mass flow rate
    # ---------------------------------------------------------
    num_parallel_channels_float = N_c / num_cell_seg
    num_parallel_channels = int(round(num_parallel_channels_float))


    if not np.isclose(
        num_parallel_channels,
        num_parallel_channels_float,
    ):
        raise ValueError(
            f'num_cell_seg={num_cell_seg} does not produce '
            'an integer channel count.'
        )
    # Cold-plate width varies with the scanned channel count
    W_plate = (num_parallel_channels * W_channel+(num_parallel_channels + 1) * S_channel)

    # The parameter scan uses the total system mass flow rate.
    # The 1D thermal solver uses the mass flow rate through one channel.
    m_dot_channel = m_dot_total / num_parallel_channels
    
    # ---------------------------------------------------------
    # Fixed rectangular-channel geometry and heat transfer
    # ---------------------------------------------------------
    A_cool_cs = W_channel* H_channel
    Dh = 2.0 * W_channel * H_channel / (W_channel + H_channel)
    A_HT_seg = (W_channel + 2.0 * H_channel) * L_channel / num_seg

    u_water = (
        m_dot_channel
        / (water_props['rho_water'] * A_cool_cs)
    )

    Re_water = (
        water_props['rho_water']
        * u_water
        * Dh
        / water_props['mu_water']
    )

    Nu_water = BTMS_model.liquid_nusselt_number_rect_channel(W_channel, H_channel)

    h_water = Nu_water * water_props['k_water'] / Dh
    R_plate = delta_plate_coolant / k_plate
    htc_global = 1.0 / (1.0 / h_water + R_plate)
    htc_cool = np.ones(num_seg) * htc_global

    # Rectangular-channel Darcy friction factor
    f_water = BTMS_model.cal_friction_factor_rect_channel(Re_water, W_channel, H_channel)

    # The pressure drop is calculated using one representative channel,
    # while pump power is calculated using the total system mass flow rate
    pump_args = {
        'fluid_cool': fluid,
        'rho_cool': water_props['rho_water'],
        'mu_cool': water_props['mu_water'],
        'A_cool_cs': A_cool_cs,
        'D_channel': Dh,
        'L_channel': L_channel,

        'm_dot_total': m_dot_total,
        'num_channel': num_parallel_channels,

        'pump_efficiency': pump_efficiency,
        'K_minor': K_minor,
        'operation_time': pump_operation_time,
    }

    pump_results = BTMS_model.cal_btms_aux_power(pump_args, friction_factor=f_water)
    P_pump = float(pump_results['P_aux_W'])
    E_pump_cumulative = float(pump_results['E_aux_J'])

    # The Reynolds number used in the pump calculation should be
    # identical to the Reynolds number used for heat transfer.
    if not np.isclose(
        pump_results['Re'],
        Re_water,
        rtol=1e-12,
        atol=0.0,
    ):
        raise RuntimeError(
            'Inconsistent Reynolds number in pump-power calculation.'
        )

    # ---------------------------------------------------------
    # Full liquid-cooling BTMS mass
    # ---------------------------------------------------------
    mass_args = {
        'fluid_cool': fluid,
        'rho_cool': water_props['rho_water'],
        'rho_plate': rho_plate,
        'L_channel': L_channel,
        'W_plate': W_plate,
        'A_cool_cs': A_cool_cs,
        'H_channel': H_channel,
        'H_bottom':  delta_plate_coolant,
        'num_channel': num_parallel_channels,

        'm_pump': m_pump,
        'm_pipe': m_pipe,
        'return_components': True,
    }

    mass_results = BTMS_model.cal_btms_mass(mass_args)
    mtotal = float(mass_results['m_BTMS_kg'])

    mass_component_sum = (
        mass_results['m_plate_kg']
        + mass_results['m_coolant_kg']
        + mass_results['m_pump_kg']
        + mass_results['m_pipe_kg']
    )

    if not np.isclose(
        mtotal,
        mass_component_sum,
        rtol=1e-12,
        atol=1e-12,
    ):
        raise RuntimeError(
            'Inconsistent component sum in BTMS mass calculation.'
        )

    # Initial battery and coolant temperatures are equal to
    # the scanned inlet-water temperature.
    T_bat = np.ones(num_seg) * T_water
    T_cool = np.ones(num_seg) * T_water

    T_bat_history = np.zeros((SIM_TIME_S + 1, num_seg))
    T_water_history = np.zeros((SIM_TIME_S + 1, num_seg))

    T_bat_history[0, :] = T_bat
    T_water_history[0, :] = T_cool

    for k in range(SIM_TIME_S):
        t = time_s[k]
        is_cool = True

        # The thermal solver represents one parallel cooling channel.
        # Therefore, it uses the per-channel coolant velocity and the
        # equivalent number of cells represented by that channel.
        args = {
            'num_seg': num_seg,
            'num_seg_bat': num_seg,
            'dt': dt,
            'T_cool_pre': T_cool,
            'T_bat_pre': T_bat,
            'u_cool_in': u_water,
            'p_cool': p_water,
            'fluid_cool': fluid,
            'A_HT_seg': A_HT_seg,
            'A_cool_cs': A_cool_cs,
            'm_bat': m_bat,
            'cp_bat': cp_bat,
            'D_bat': D_bat,
            'T_cool_in': T_water,
            'T_cool_out': max(
                float(T_cool[-1]),
                T_water,
            ),
            'is_cool': is_cool,
            'htc_cool': htc_cool,
            'cp_cool': water_props['cp_water'],
            'rho_cool': water_props['rho_water'],
            'Q_gen': power_generation_data[k],
            'num_cell_seg': num_cell_seg,
            'debug': False,
        }

        try:
            with contextlib.redirect_stdout(io.StringIO()):
                T_dist = (
                    BTMS_model
                    .solve_coolant_temperature_distribution(
                        args,
                        tol=solver_tol,
                        maxiter=solver_maxiter,
                        debug=False,
                    )
                )

        except Exception as e:
            print('\nSolver failed inside run_one_case.')
            print(f'case_id = {case_id}')
            print(
                f'm_dot_total = {m_dot_total} kg/s'
            )
            print(
                f'm_dot_channel = {m_dot_channel} kg/s'
            )
            print(
                f'num_parallel_channels = '
                f'{num_parallel_channels}'
            )
            print(f'T_water_in = {T_water} degC')
            print(f'Dh = {Dh * 1e3} mm')
            print(f'num_cell_seg = {num_cell_seg}')
            print(f'time step k = {k}')
            print(f't = {t:.1f} s')
            print(f'is_cool = {is_cool}')
            print(
                f'Q_gen = {power_generation_data[k]} W'
            )
            print(f'u_water = {u_water}')
            print(f'Re_water = {Re_water}')
            print(f'Nu_water = {Nu_water}')
            print(f'h_water = {h_water}')
            print(f'htc_global = {htc_global}')
            print(f'A_HT_seg = {A_HT_seg}')
            print(f'A_cool_cs = {A_cool_cs}')
            print(f'P_pump = {P_pump}')
            print(
                f'E_pump_cumulative = '
                f'{E_pump_cumulative}'
            )
            print(f'mtotal = {mtotal}')
            print(
                'T_bat_min/max before solve = '
                f'{np.min(T_bat)}, {np.max(T_bat)}'
            )
            print(
                'T_cool_min/max before solve = '
                f'{np.min(T_cool)}, {np.max(T_cool)}'
            )
            print(f'solver_tol = {solver_tol}')
            print(f'solver_maxiter = {solver_maxiter}')
            print(f'error = {repr(e)}')
            raise

        if not np.all(np.isfinite(T_dist)):
            print('\nSolver returned non-finite values.')
            print(f'case_id = {case_id}')
            print(
                f'm_dot_total = {m_dot_total} kg/s'
            )
            print(
                f'm_dot_channel = {m_dot_channel} kg/s'
            )
            print(
                f'num_parallel_channels = '
                f'{num_parallel_channels}'
            )
            print(f'T_water_in = {T_water} degC')
            print(f'Dh = {Dh * 1e3} mm')
            print(f'num_cell_seg = {num_cell_seg}')
            print(f'time step k = {k}')
            print(f't = {t:.1f} s')
            print(
                'T_dist_min/max = '
                f'{np.nanmin(T_dist)}, '
                f'{np.nanmax(T_dist)}'
            )
            raise FloatingPointError(
                'Non-finite values detected in T_dist'
            )

        T_cool = T_dist[:num_seg]
        T_bat = T_dist[num_seg:]

        T_bat_history[k + 1, :] = T_bat
        T_water_history[k + 1, :] = T_cool

    Tmax = np.max(T_bat_history, axis=1)
    Tmin = np.min(T_bat_history, axis=1)
    DeltaT = Tmax - Tmin
    Twater_out = T_water_history[:, -1]
    cruise_end_time = next(
        t1
        for stage_name, _, t1 in mission_stages
        if stage_name == 'cruise'
    )
    def idx(t):
        return int(round(t / dt))
    
    row = {
        'case_id': case_id,

        
        # Total system mass flow rate: parameter-scan variable.
        'm_dot_total_kg_s': float(m_dot_total),

        # Per-channel mass flow rate: thermal-solver variable.
        'm_dot_channel_kg_s': float(m_dot_channel),

        'T_water_in_C': float(T_water),
        'Dh_mm': float(Dh * 1e3),
        'num_cell_seg': float(num_cell_seg),
        'num_parallel_channels': num_parallel_channels,
        'u_water_m_s': float(u_water),
        'Re_water': float(Re_water),
        'P_pump': P_pump,
        'E_pump_cumulative': E_pump_cumulative,
        'm_plate_kg': float(
            mass_results['m_plate_kg']
        ),
        'm_coolant_kg': float(
            mass_results['m_coolant_kg']
        ),
        'mtotal': mtotal,
        'Tmax_mission_C': float(np.max(Tmax)),
        'DeltaT_mission_max_C': float(
            np.max(DeltaT)
        ),
        'Twater_out_cruise_end_C': float(
            Twater_out[idx(cruise_end_time)]
        ),
    }

    for stage_name, t0, t1 in mission_stages:
        row[f'dTmax_{stage_name}_C'] = float(
            Tmax[idx(t1)] - Tmax[idx(t0)]
        )

    return row

In [ ]:
# ============================================================
# Cell 7 - Run SCAN1 Cases and Export Results
# ============================================================


def format_excel_table(xlsx_path):
    """Apply the original Times New Roman table style to the exported Excel file."""

    wb = load_workbook(xlsx_path)
    ws = wb.active
    ws.title = 'Parameter scan'

    body_font = Font(name='Times New Roman',size=10,)
    header_font = Font(name='Times New Roman',size=10,bold=True,)
    alignment = Alignment(horizontal='center',vertical='center',)

    thin = Side(style='thin')
    border = Border(left=thin,right=thin,top=thin,bottom=thin,)

    for row in ws.iter_rows():
        for cell in row:

            cell.font = (
                header_font
                if cell.row == 1
                else body_font
            )

            cell.alignment = alignment
            cell.border = border

            if (
                cell.row > 1
                and isinstance(cell.value, float)
            ):
                cell.number_format = '0.0000'

    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions

    for col_idx, column_cells in enumerate(
        ws.columns,
        start=1,
    ):
        max_len = max(
            len(str(cell.value))
            if cell.value is not None
            else 0
            for cell in column_cells
        )

        ws.column_dimensions[
            get_column_letter(col_idx)
        ].width = min(
            max(max_len + 2, 12),
            28,
        )

    wb.save(xlsx_path)


# ============================================================
# Run all parameter-scan cases
# ============================================================

summary_rows = []

for n, case in enumerate(
    scan_cases,
    start=1,
):

    num_parallel_channels = int(
        round(
            N_c
            / case['num_cell_seg']
        )
    )

    m_dot_channel = (
        case['m_dot_total']
        / num_parallel_channels
    )

    print(
        f"Running {n}/{len(scan_cases)}: "
        f"{case['case_id']}, "
        f"m_dot_total={case['m_dot_total']} kg/s, "
        f"m_dot_channel={m_dot_channel:.6f} kg/s, "
        f"T_water={case['T_water']} degC, "
        f"num_parallel_channels={num_parallel_channels}, "
        f"num_cell_seg={case['num_cell_seg']}"
    )

    try:
        result_row = run_one_case(
            **case
        )

        summary_rows.append(
            result_row
        )

    except Exception as e:

        print(
            '\nParameter scan stopped because '
            'one case failed.'
        )

        print(
            f'failed index = '
            f'{n}/{len(scan_cases)}'
        )

        print(
            f"failed case_id = "
            f"{case['case_id']}"
        )

        print(
            f"failed m_dot_total = "
            f"{case['m_dot_total']} kg/s"
        )

        print(
            f'failed m_dot_channel = '
            f'{m_dot_channel} kg/s'
        )

        print(
            f"failed T_water_in = "
            f"{case['T_water']} degC"
        )

        print(
            f'failed num_parallel_channels = '
            f'{num_parallel_channels}'
        )

        print(
            f"failed num_cell_seg = "
            f"{case['num_cell_seg']}"
        )

        print(
            f'error = {repr(e)}'
        )

        raise


# ============================================================
# Build the final parameter-scan table
# ============================================================

parameter_scan_table = pd.DataFrame(summary_rows)

column_order = [
    'case_id',

    # Scanned parameters
    'T_water_in_C',
    'm_dot_total_kg_s',
    'num_cell_seg',

    # Derived channel parameters
    'num_parallel_channels',
    'm_dot_channel_kg_s',
    'Dh_mm',

    # Flow indicators
    'u_water_m_s',
    'Re_water',

    # Pump performance
    'P_pump',
    'E_pump_cumulative',

    # BTMS mass
    'm_plate_kg',
    'm_coolant_kg',
    'mtotal',

    # Full-mission thermal indicators
    'Tmax_mission_C',
    'DeltaT_mission_max_C',
    'Twater_out_cruise_end_C',

    # Stage-wise battery-temperature changes
    'dTmax_takeoff_C',
    'dTmax_climb_C',
    'dTmax_transition1_C',
    'dTmax_cruise_C',
    'dTmax_transition2_C',
    'dTmax_descent_C',
    'dTmax_hover_C',
    'dTmax_landing_C',
]

parameter_scan_table = (
    parameter_scan_table[
        column_order
    ]
)


# ============================================================
# Export the official Excel table
# ============================================================

xlsx_path = RESULT_XLSX_PATH

parameter_scan_table.to_excel(xlsx_path, index=False)
format_excel_table(xlsx_path)

print(f'Saved official Excel: {xlsx_path}')

print(
    'Only one table file is generated '
    'for this case.'
)

parameter_scan_table.head()
           



Running 1/144: C0003_S001, m_dot_total=0.045 kg/s, m_dot_channel=0.003750 kg/s, T_water=20.0 degC, num_parallel_channels=12, num_cell_seg=1.3333333333333333
Running 2/144: C0003_S002, m_dot_total=0.045 kg/s, m_dot_channel=0.002812 kg/s, T_water=20.0 degC, num_parallel_channels=16, num_cell_seg=1.0
Running 3/144: C0003_S003, m_dot_total=0.045 kg/s, m_dot_channel=0.002250 kg/s, T_water=20.0 degC, num_parallel_channels=20, num_cell_seg=0.8
Running 4/144: C0003_S004, m_dot_total=0.045 kg/s, m_dot_channel=0.001875 kg/s, T_water=20.0 degC, num_parallel_channels=24, num_cell_seg=0.6666666666666666
Running 5/144: C0003_S005, m_dot_total=0.045 kg/s, m_dot_channel=0.001607 kg/s, T_water=20.0 degC, num_parallel_channels=28, num_cell_seg=0.5714285714285714
Running 6/144: C0003_S006, m_dot_total=0.045 kg/s, m_dot_channel=0.001406 kg/s, T_water=20.0 degC, num_parallel_channels=32, num_cell_seg=0.5
Running 7/144: C0003_S007, m_dot_total=0.05 kg/s, m_dot_channel=0.004167 kg/s, T_water=20.0 degC, num_p

,case_id,T_water_in_C,m_dot_total_kg_s,num_cell_seg,num_parallel_channels,m_dot_channel_kg_s,Dh_mm,u_water_m_s,Re_water,P_pump,...,DeltaT_mission_max_C,Twater_out_cruise_end_C,dTmax_takeoff_C,dTmax_climb_C,dTmax_transition1_C,dTmax_cruise_C,dTmax_transition2_C,dTmax_descent_C,dTmax_hover_C,dTmax_landing_C
0,C0003_S001,20.0,0.045,1.333333,12,0.003750,4.363636,0.156531,680.731637,0.013402,...,1.085045,20.475053,0.415090,1.475418,1.115929,-0.479069,-0.755113,0.508139,12.459443,0.376525
1,C0003_S002,20.0,0.045,1.000000,16,0.002812,4.363636,0.117398,510.548728,0.010052,...,1.423821,20.469645,0.414166,1.453403,0.932040,-0.808965,-0.714706,0.515965,11.829531,0.339079
2,C0003_S003,20.0,0.045,0.800000,20,0.002250,4.363636,0.093918,408.438982,0.008041,...,1.775868,20.465828,0.413397,1.435172,0.786187,-0.964211,-0.682568,0.521732,11.317774,0.309541
3,C0003_S004,20.0,0.045,0.666667,24,0.001875,4.363636,0.078265,340.365818,0.006701,...,2.124363,20.463283,0.412765,1.420130,0.667734,-1.041080,-0.657209,0.526117,10.895990,0.285642
4,C0003_S005,20.0,0.045,0.571429,28,0.001607,4.363636,0.067085,291.742130,0.005744,...,2.458603,20.461535,0.412251,1.407779,0.569360,-1.079064,-0.636819,0.529654,10.543664,0.265857


## SCAN2- Geometric Parameter Scan - the 2nd stage scan

SCAN2 evaluates the effects of channel width, channel height, and channel length under a fixed operating condition.

### Required cells
Before running SCAN2, run the following cells:

* **Cell 0** - Import packages and BTMS model
* **Cell 1** - File and output settings
* **Cell 2** - Heat-generation data
* **Cell 3** - Model and solver settings
* **Cell 6** - Define `run_one_case()`
The SCAN1-specific cells below are not required for SCAN2:

* **Cell 4**
* **Cell 5**
* **Cell 7**
After that, run **Cell 8 onward**.

### Execution order
Cell 0 → Cell 1 → Cell 2 → Cell 3 → Cell 6 → Cell 8 → Cell 9 → Cell 10 → Cell 11


In [ ]:
# ============================================================
# Cell 8 - Second-Stage Geometric Parameter-Scan Settings
# ============================================================

T_water = 20.0                  # degC
m_dot_total = 0.060             # kg/s
num_parallel_channels = 20

num_cell_seg = N_c / num_parallel_channels


# ------------------------------------------------------------
# Geometric parameter-scan ranges
# ------------------------------------------------------------

W_channel_list = np.array([
    6.0,
    7.0,
    8.0,
    9.0,
    10.0,
], dtype=float) * 1e-3


H_channel_list = np.array([
    2.0,
    3.0,
    4.0,
    5.0,
], dtype=float) * 1e-3


L_channel_list = np.array([
    0.380,
    0.390,
    0.400,
    0.410,
    0.420,
], dtype=float)


print('Second-stage geometric scan settings loaded.')

print(f'T_water = {T_water:.1f} degC')
print(f'm_dot_total = {m_dot_total:.3f} kg/s')
print(f'num_parallel_channels = {num_parallel_channels}')
print(f'num_cell_seg = {num_cell_seg:.4f}')

print(f'W_channel scan = {W_channel_list * 1e3} mm')
print(f'H_channel scan = {H_channel_list * 1e3} mm')
print(f'L_channel scan = {L_channel_list * 1e3} mm')



Second-stage geometric scan settings loaded.
T_water = 20.0 degC
m_dot_total = 0.060 kg/s
num_parallel_channels = 20
num_cell_seg = 0.8000
W_channel scan = [ 6.  7.  8.  9. 10.] mm
H_channel scan = [2. 3. 4. 5.] mm
L_channel scan = [380. 390. 400. 410. 420.] mm


In [ ]:
# ============================================================
# Cell 9 - Calculate Coolant Properties for SCAN2
# ============================================================

T_water_K = T_water + 273.15

mu_water = CP.PropsSI('V', 'T', T_water_K, 'P', p_water, fluid)
rho_water = CP.PropsSI('D', 'T', T_water_K, 'P', p_water, fluid)
cp_water = CP.PropsSI('C', 'T', T_water_K, 'P', p_water, fluid)
k_water = CP.PropsSI('L', 'T', T_water_K, 'P', p_water, fluid)
Pr_water = cp_water * mu_water / k_water

water_props_cache = {
    float(T_water): {
        'mu_water': mu_water,
        'rho_water': rho_water,
        'cp_water': cp_water,
        'k_water': k_water,
        'Pr_water': Pr_water,
    }
}

print(f'Water properties calculated at T_water = {T_water:.1f} degC.')

Water properties calculated at T_water = 20.0 degC.


In [8]:
# ============================================================
# Cell 10 - Run geometric parameter scan
# ============================================================

scan2_rows = []
case_id = 0
total_cases = len(W_channel_list) * len(H_channel_list) * len(L_channel_list)

for W in W_channel_list:
    for H in H_channel_list:
        for L in L_channel_list:
            case_id += 1

            W_channel = float(W)
            H_channel = float(H)
            L_channel = float(L)

            print(f'Running {case_id}/{total_cases}: W={W_channel*1e3:.1f} mm, H={H_channel*1e3:.1f} mm, L={L_channel*1e3:.1f} mm')

            result_row = run_one_case(case_id, T_water, m_dot_total, num_cell_seg)

            result_row['W_channel_mm'] = W_channel * 1e3
            result_row['H_channel_mm'] = H_channel * 1e3
            result_row['L_channel_mm'] = L_channel * 1e3

            scan2_rows.append(result_row)

print(f'Geometry scan completed: {len(scan2_rows)} cases.')
W_channel = LIQUID_SETTINGS['W_channel']
H_channel = LIQUID_SETTINGS['H_channel']
L_channel = LIQUID_SETTINGS['L_channel']

Running 1/100: W=6.0 mm, H=2.0 mm, L=380.0 mm
Running 2/100: W=6.0 mm, H=2.0 mm, L=390.0 mm
Running 3/100: W=6.0 mm, H=2.0 mm, L=400.0 mm
Running 4/100: W=6.0 mm, H=2.0 mm, L=410.0 mm
Running 5/100: W=6.0 mm, H=2.0 mm, L=420.0 mm
Running 6/100: W=6.0 mm, H=3.0 mm, L=380.0 mm
Running 7/100: W=6.0 mm, H=3.0 mm, L=390.0 mm
Running 8/100: W=6.0 mm, H=3.0 mm, L=400.0 mm
Running 9/100: W=6.0 mm, H=3.0 mm, L=410.0 mm
Running 10/100: W=6.0 mm, H=3.0 mm, L=420.0 mm
Running 11/100: W=6.0 mm, H=4.0 mm, L=380.0 mm
Running 12/100: W=6.0 mm, H=4.0 mm, L=390.0 mm
Running 13/100: W=6.0 mm, H=4.0 mm, L=400.0 mm
Running 14/100: W=6.0 mm, H=4.0 mm, L=410.0 mm
Running 15/100: W=6.0 mm, H=4.0 mm, L=420.0 mm
Running 16/100: W=6.0 mm, H=5.0 mm, L=380.0 mm
Running 17/100: W=6.0 mm, H=5.0 mm, L=390.0 mm
Running 18/100: W=6.0 mm, H=5.0 mm, L=400.0 mm
Running 19/100: W=6.0 mm, H=5.0 mm, L=410.0 mm
Running 20/100: W=6.0 mm, H=5.0 mm, L=420.0 mm
Running 21/100: W=7.0 mm, H=2.0 mm, L=380.0 mm
Running 22/100: W=7.0 

In [ ]:
# ============================================================
# Cell 11 - Build and export SCAN2 result table
# ============================================================

scan2_table = pd.DataFrame(scan2_rows)

column_order = [
    'case_id',

    # Scanned parameters
    'W_channel_mm',
    'H_channel_mm',
    'L_channel_mm',

    # Derived channel parameters
    'num_parallel_channels',
    'm_dot_channel_kg_s',
    'Dh_mm',

    # Flow indicators
    'u_water_m_s',
    'Re_water',

    # Pump performance
    'P_pump',
    'E_pump_cumulative',

    # BTMS mass
    'm_plate_kg',
    'm_coolant_kg',
    'mtotal',

    # Full-mission thermal indicators
    'Tmax_mission_C',
    'DeltaT_mission_max_C',
    'Twater_out_cruise_end_C',

    # Stage-wise battery-temperature changes
    'dTmax_takeoff_C',
    'dTmax_climb_C',
    'dTmax_transition1_C',
    'dTmax_cruise_C',
    'dTmax_transition2_C',
    'dTmax_descent_C',
    'dTmax_hover_C',
    'dTmax_landing_C',
]

scan2_table = scan2_table[column_order]

SCAN_XLSX_PATH = os.path.join(OUTPUT_DIR, 'C0003R02_PARK25_LC_1D_SCAN.xlsx')

with pd.ExcelWriter(
    SCAN_XLSX_PATH,
    engine='openpyxl',
    mode='a',
    if_sheet_exists='replace',
) as writer:

    scan2_table.to_excel(
        writer,
        sheet_name='Geometry scan_stage2',
        index=False,
    )

wb = load_workbook(SCAN_XLSX_PATH)
ws = wb['Geometry scan_stage2']

body_font = Font(name='Times New Roman', size=10)
header_font = Font(name='Times New Roman', size=10, bold=True)
alignment = Alignment(horizontal='center', vertical='center')
thin = Side(style='thin')
border = Border(left=thin, right=thin, top=thin, bottom=thin)

for row in ws.iter_rows():
    for cell in row:
        cell.font = header_font if cell.row == 1 else body_font
        cell.alignment = alignment
        cell.border = border
        if cell.row > 1 and isinstance(cell.value, float): cell.number_format = '0.0000'

ws.freeze_panes = 'A2'
ws.auto_filter.ref = ws.dimensions

for col_idx, column_cells in enumerate(ws.columns, start=1):
    max_len = max(len(str(cell.value)) if cell.value is not None else 0 for cell in column_cells)
    ws.column_dimensions[get_column_letter(col_idx)].width = min(max(max_len + 2, 12), 28)

wb.save(SCAN_XLSX_PATH)

print(f'Saved SCAN2 Excel: {SCAN_XLSX_PATH}')
print(f'Total cases: {len(scan2_table)}')

scan2_table.head()

Saved SCAN2 Excel: d:\Users\liu\Desktop\eBATS_scanned_2\out\C0003\C0003R02_PARK25_LC_1D_SCAN2.xlsx
Total cases: 100


,case_id,W_channel_mm,H_channel_mm,L_channel_mm,num_parallel_channels,m_dot_channel_kg_s,Dh_mm,u_water_m_s,Re_water,P_pump,...,DeltaT_mission_max_C,Twater_out_cruise_end_C,dTmax_takeoff_C,dTmax_climb_C,dTmax_transition1_C,dTmax_cruise_C,dTmax_transition2_C,dTmax_descent_C,dTmax_hover_C,dTmax_landing_C
0,1,6.0,2.0,380.0,20,0.003,3.0,0.250449,748.8048,0.062189,...,1.473774,20.347991,0.412518,1.414937,0.649282,-1.015736,-0.652632,0.525733,10.802118,0.282402
1,2,6.0,2.0,390.0,20,0.003,3.0,0.250449,748.8048,0.063826,...,1.512862,20.347723,0.412401,1.412221,0.630049,-1.023572,-0.648477,0.526337,10.730816,0.278570
2,3,6.0,2.0,400.0,20,0.003,3.0,0.250449,748.8048,0.065462,...,1.551804,20.347469,0.412286,1.409560,0.611318,-1.030428,-0.644440,0.526915,10.661151,0.274842
3,4,6.0,2.0,410.0,20,0.003,3.0,0.250449,748.8048,0.067099,...,1.590577,20.347229,0.412174,1.406952,0.593071,-1.036397,-0.640516,0.527471,10.593072,0.271215
4,5,6.0,2.0,420.0,20,0.003,3.0,0.250449,748.8048,0.068735,...,1.629158,20.347002,0.412064,1.404397,0.575288,-1.041564,-0.636700,0.528004,10.526528,0.267684
